# Electronic coupling

Cleaned and organized for reproducible execution from the repository root.


In [ ]:
# File: electron_electron_coupling_3D.py
from qutip import tensor, sigmaz, sigmap, sigmam, qeye, mesolve, basis
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
# Function: two-electron (two-qubit) coupling dynamics
def ee_integrate(wa, wb, J, gamma_a, gamma_b, psi0, tlist):
    """
    Simulate two two-level systems (electrons) coupled via XY interaction.
    H = wa/2 * sigma_z^A + wb/2 * sigma_z^B + J*(sigma_+^A sigma_-^B + sigma_-^A sigma_+^B)
    Includes local relaxation.
    """
    # Pauli operators
    sa = tensor(sigmaz(), qeye(2))   # sigma_z on qubit A
    sb = tensor(qeye(2), sigmaz())   # sigma_z on qubit B
    spa = tensor(sigmap(), qeye(2))  # sigma_+ on A
    sma = tensor(sigmam(), qeye(2))  # sigma_- on A
    spb = tensor(qeye(2), sigmap())  # sigma_+ on B
    smb = tensor(qeye(2), sigmam())  # sigma_- on B

    # Hamiltonian
    H = (wa / 2) * sa + (wb / 2) * sb + J * (spa * smb + sma * spb)

    # Collapse operators (relaxation)
    c_ops = []
    if gamma_a > 0:
        c_ops.append(np.sqrt(gamma_a) * sma)
    if gamma_b > 0:
        c_ops.append(np.sqrt(gamma_b) * smb)

    # Solve master equation
    result = mesolve(H, psi0, tlist, c_ops, [])
    return result

In [ ]:
# Example usage
if __name__ == '__main__':
    wa, wb = 1.0 * 2 * np.pi, 1.2 * 2 * np.pi
    J = 0.1 * 2 * np.pi
    gamma_a, gamma_b = 0.05, 0.05
    tlist = np.linspace(0, 10, 200)

    # Initial state: Electron A excited, Electron B ground
    psi0 = tensor(basis(2, 1), basis(2, 0))
    result = ee_integrate(wa, wb, J, gamma_a, gamma_b, psi0, tlist)

    # Calculate populations
    ex_a = [np.real((psi.dag() * psi).full()[1,1]) for psi in result.states]  # population of |1> for A
    ex_b = [np.real((psi.dag() * psi).full()[1,1]) for psi in result.states]  # same for B; or use expect operators

In [ ]:
    # Plot
    plt.plot(tlist, ex_a, label='Electron A excited')
    plt.plot(tlist, ex_b, label='Electron B excited')
    plt.xlabel('Time')
    plt.ylabel('Population')
    plt.legend()
    plt.show()

In [ ]:
# Example usage and 3D animation
def main():
    # Parameters
    wa, wb = 1.0 * 2 * np.pi, 1.2 * 2 * np.pi
    J = 0.1 * 2 * np.pi
    gamma_a, gamma_b = 0.05, 0.05
    tlist = np.linspace(0, 10, 200)

    # Initial state: A excited, B ground
    psi0 = tensor(basis(2, 1), basis(2, 0))
    result = ee_integrate(wa, wb, J, gamma_a, gamma_b, psi0, tlist)

    # Compute populations
    ex_a = [np.real((state.dag() * state).full()[1,1]) for state in result.states]
    ex_b = [np.real((state.dag() * state).full()[3,3]) for state in result.states]
    # Note: index 3,3 corresponds to |11> state; for two qubits, adjust accordingly or use expect

    # Setup 3D animation: time vs ex_a vs ex_b
    fig = plt.figure(figsize=(8,6))
    ax = fig.add_subplot(111, projection='3d')
    ax.set_xlabel('Time')
    ax.set_ylabel('P(A excited)')
    ax.set_zlabel('P(B excited)')
    ax.set_xlim(tlist.min(), tlist.max())
    ax.set_ylim(0,1)
    ax.set_zlim(0,1)

    # Plot full trajectory
    ax.plot(tlist, ex_a, ex_b, color='gray', alpha=0.5)

    # Marker for animation
    marker, = ax.plot([tlist[0]], [ex_a[0]], [ex_b[0]], 'o', color='red')

    def update(frame):
        marker.set_data([tlist[frame]], [ex_a[frame]])
        marker.set_3d_properties([ex_b[frame]])
        return marker,

    anim = animation.FuncAnimation(fig, update, frames=len(tlist), interval=50, blit=True)
    plt.tight_layout()

    # Display animation in Jupyter
    from IPython.display import HTML
    display(HTML(anim.to_jshtml()))


if __name__ == '__main__':
    main()
